## inspect4j: Preliminary Investigation

This notebook focus on experimentation to conduct preliminary investigation to deepen knownledge and understanding on `inspect4j`.

It uses the version that exist in the pypi.

Author: Boon Heng Chew <<s2831745@sms.ed.ac.uk>>

### Introduction to inspect4py

inspect4j is a python library or tool that parse Java code repositories to extract metadata, documentation, structure, and dependencies. It automates the analysis of

- Function, classes, and source code
- requirements (and licenses)
- repository structure (directory trees)
- control flow

In [3]:
from platform import python_version

# show environment
print(f"Python environment: {python_version()}")

Python environment: 3.12.4


### Setup

inspect4j is installed and manage using uv in the virtual environment.

```sh
# in this directory, run the command below to add it as a dependency
uv add inspect4j
```

In [5]:
# Shows the tool's command arguments
!inspect4j --help

zsh:1: command not found: inspect4j


#### Finding 1:

inspect4j is not functionally at the same capability level of inspect4py. It does not support the following:

- `-si:` software invocation
- `-sc:` source code (generate for each ast node)
- `-ast:` generate the abstract syntax tree
- `-cf:` control flow where the call graph for each file in target repository is generated
- `-df:` data flow graph (not working with fastapi)

### Testing with inspect4j

Perform experimentation with `inspect4j` to understand how it works and the output from the tool. 

To test it with a simple java repository as samples. 

The tool will be run with the following command and flags.

```sh
inspect4j -i input_directory -o output_directory -r -ld -cl -dt -rm -md -html
```

It will generate the following:
- `-r:` requirements (dependencies)
- `-ld:` license detection
- `-cl:` call list
- `-dt:` directory tree (file directory tree from root path of repo)
- `-rm:` readme (extract all readme files)
- `-md:` metadata (github repository metadata using github api. input is `.git`)
- `-html:` also generate HTML output


In [8]:
# Download sample java repository
!git clone https://github.com/BuntyRaghani/spring-boot-3-hello-world.git

Cloning into 'spring-boot-3-hello-world'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 55 (delta 10), reused 9 (delta 8), pack-reused 41 (from 2)
Receiving objects: 100% (55/55), 7.28 KiB | 1.82 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [9]:
# Parse repository and run analysis using inspect4j
!inspect4j -i spring-boot-3-hello-world -o spring-boot-3-hello-world-output -r -ld -cl -dt -rm -md -html

zsh:1: command not found: inspect4j


In [10]:
# show the main extracted metadata (i.e. directory_info.json)
!cat spring-boot-3-hello-world-output/directory_info.json

{
  "spring-boot-3-hello-world-output/spring-boot-3-hello-world/src/test/java/com/example/helloworld": [
    {
      "file": {
        "path": "/Users/boonheng/Documents/msc_studies/dissertation/experiments/inspect4j-experiment/spring-boot-3-hello-world/src/test/java/com/example/helloworld/HelloWorldApplicationTests.java",
        "fileNameBase": "HelloWorldApplicationTests",
        "extension": "java"
      },
      "package": {
        "name": "com.example.helloworld"
      },
      "dependencies": [
        {
          "import": "HelloWorldController",
          "type": "internal",
          "type_element": "class",
          "from_module": "com.example.helloworld.controller"
        },
        {
          "import": "Test",
          "type": "external",
          "type_element": "class",
          "from_module": "org.junit.jupiter.api"
        },
        {
          "import": "Autowired",
          "type": "external",
          "type_element": "class",
          "from_module": "org

#### Finding 2:

inspect4j does not generate a separate requirement/dependany file for library dependencies. It is part of the main `directory-info.json`

In addition, It fail to extract the right library version if they are defined in parent pom.xml (e.g. standard java springboot app)

#### Finding 3:

Overall, inspect4j is still the closest varient to inspect4py after running the tools and looking at the past dissertation report on these tools.

- inspect4c and inspect4c++ are both structurally different from inspect4py and much more complex

#### Finding 4 (from paper study):

`javalang` handles Java 8 syntax correctly, including lambda expressions and generics. However, it fails to parse Java 9+ language features (leading to next test in the notebook). Running it against a class using a `record` (Java 16+) produces a `JavaSyntaxError` with no output generated for that file.

- inspect4c and inspect4c++ are both structurally different from inspect4py and much more complex

### Testing javalang's Java 17+ Limitation

`javalang` targets Java 8 syntax. To verify its hard failure on modern Java, a minimal Java 17 project is created locally using a `record` class — a feature finalized in Java 16 and common in Java 17+ codebases.

In [15]:
import os

# Review the java source containing the new syntax
with open("java17-test/src/Point.java", "w") as f:
    f.write(java_record)

print("java17-test/src/Point.java: \n")
print(open("java17-test/src/Point.java").read())

NameError: name 'java_record' is not defined

In [ ]:
# Run inspect4j on the Java 17 record project
!inspect4j -i java17-test -o java17-test-output -cl -dt 2>&1

In [ ]:
import json, os

output_path = "java17-test-output"
if os.path.exists(output_path):
    for root, dirs, files in os.walk(output_path):
        for fname in files:
            if fname.endswith(".json"):
                fpath = os.path.join(root, fname)
                print(f"=== {fpath} ===")
                with open(fpath) as fp:
                    data = json.load(fp)
                print(json.dumps(data, indent=2))
else:
    print("No output generated — inspect4j failed to parse the file.")

#### Finding 5: javalang fails to parse Java 17+ record syntax

`inspect4j` (via `javalang`) cannot parse the `record` keyword introduced in Java 16. Running inspect4j on a minimal Java 17 record class produces a `ParseException` and no output for the file.

This confirms the limitation in Finding 4 is a hard failure, not just missing metadata. Any modern Java codebase using records, sealed classes, or other post-Java 8 syntax will be incompatible with `inspect4j` as-is.